# Lecture 1 — Text Preprocessing and Tokenization

**Course:** Large Language Models  
**Topic:** From raw text to token IDs  
**Tools:** Python, PyTorch, Hugging Face `transformers` and `tokenizers`

## Learning objectives

By the end of this notebook, you should be able to:

1. Explain the role of text preprocessing and tokenization.
2. Compare character-, word-, and subword-level tokenization.
3. Implement a simplified Byte Pair Encoding (BPE) algorithm.
4. Explain the differences between BPE, WordPiece, and Unigram.
5. Load, inspect, encode, decode, and batch text with Hugging Face tokenizers.
6. Train a small byte-level BPE tokenizer.
7. Explain how token IDs are mapped to embedding vectors.

> **Important distinction:** A tokenizer converts text into discrete token IDs. It does not create semantic embeddings and does not train a language model.


## 0. Setup

Install the libraries in your environment. If you are using Jupyter, run the following cell once.

The pretrained-tokenizer examples download tokenizer files from the Hugging Face Hub, so those examples require an internet connection on first use.


In [ ]:
%pip install -q transformers tokenizers datasets torch

## 1. Text preprocessing

Text preprocessing consists of operations applied to raw text before it enters a modeling pipeline. Examples include Unicode normalization, whitespace normalization, HTML removal, and sometimes lowercasing.

Preprocessing is **task- and model-dependent**. Lowercasing is not always desirable: `Apple` and `apple`, for example, can refer to different things. Many pretrained tokenizers already include their own normalization rules, so applying additional transformations can change the model's expected input.

### Example: conservative normalization


In [ ]:
import re
import unicodedata

raw_text = "   HELLO,    WORLD!   "

# Canonical Unicode normalization
text = unicodedata.normalize("NFC", raw_text)

# Remove leading/trailing whitespace
text = text.strip()

# Replace consecutive whitespace with one space
text = re.sub(r"\\s+", " ", text)

print(repr(text))

### What does this code do?

- `unicodedata.normalize("NFC", text)` applies Unicode canonical composition. Visually identical strings can have different underlying Unicode representations.
- `.strip()` removes whitespace at the beginning and end.
- `re.sub(r"\\s+", " ", text)` replaces each run of whitespace with one ordinary space.
- We deliberately do **not** lowercase the text. Whether to lowercase depends on the model and task.

Preprocessing should preserve information that may be useful to the model.


## 2. Tokenization

**Tokenization** maps a text string into a sequence of smaller units called *tokens*. Depending on the method, a token can be a character, a word, punctuation, a subword, or a byte-level symbol.

Consider:

`I love machine learning.`

### 2.1 Character-level tokenization


In [ ]:
text = "hello"
character_tokens = list(text)

print(character_tokens)
print("Number of tokens:", len(character_tokens))

Each character is a token. For a string containing \(N\) characters, character-level tokenization produces approximately \(N\) tokens (depending on how whitespace and Unicode are represented).

**Advantages**
- Small vocabulary.
- Can represent arbitrary strings when the character inventory covers the input.

**Disadvantages**
- Long sequences.
- The model must learn word and subword patterns from character sequences.
- Long sequences increase attention computation and memory requirements.


### 2.2 Word-level tokenization

In [ ]:
text = "I love deep learning"
word_tokens = text.split()

print(word_tokens)
print("Number of tokens:", len(word_tokens))

Word-level tokenization treats each whitespace-separated word as one token in this simple example. Real word tokenizers may handle punctuation and language-specific segmentation differently.

**Advantages**
- Shorter sequences than character-level tokenization.
- Whole words are retained as units.

**Disadvantages**
- Vocabulary can become very large.
- New or rare words may be out of vocabulary (OOV).
- Morphological variants such as `play`, `playing`, and `played` may require separate entries.


### 2.3 Subword tokenization

Subword tokenization represents text using units that may be complete words or parts of words. For example, a tokenizer might split `unbelievable` into pieces such as `un`, `believ`, and `able`. The exact segmentation is learned from training data and depends on the tokenizer.

Subword tokenization is a compromise:
- Usually a smaller vocabulary than word-level tokenization.
- Usually shorter sequences than character-level tokenization.
- Can represent many unseen words by composing known subword units.

The following illustration is conceptual, not a guaranteed segmentation:


In [ ]:
word = "unbelievable"
illustrative_subwords = ["un", "believ", "able"]

print("Original:", word)
print("Illustrative segmentation:", illustrative_subwords)
print("Reconstructed:", "".join(illustrative_subwords))

## 3. Byte Pair Encoding (BPE)

BPE was originally developed as a data-compression method and was later adapted for subword tokenization.

### Core idea

1. Start with a vocabulary of small units (in this teaching example, characters).
2. Count adjacent pairs across the training corpus, weighted by word frequency.
3. Merge the most frequent adjacent pair into a new unit.
4. Repeat until the desired number of merges or vocabulary size is reached.

The pair selected at iteration \(t\) is:

\[
(a_t,b_t)=\arg\max_{(a,b)} f_t(a,b),
\]

where \(f_t(a,b)\) is the weighted frequency of adjacent pair \((a,b)\) at iteration \(t\).

The vocabulary is updated by adding the merged token:

\[
V_{t+1}=V_t\cup\{a_tb_t\}.
\]

### 3.1 A small corpus

We use four words with frequencies:


In [ ]:
corpus = {
    "hug": 10,
    "pug": 5,
    "pun": 12,
    "bun": 4,
}

# Represent each word as a tuple of characters.
vocab = {tuple(word): frequency for word, frequency in corpus.items()}

for word, frequency in vocab.items():
    print(word, ":", frequency)

### 3.2 Count adjacent pairs

For example, `hug` contributes `(h, u)` and `(u, g)`, each with weight 10. The frequency of a pair is the sum of the frequencies of all words in which that pair occurs.



In [ ]:
from collections import Counter

def get_pair_frequencies(vocab):
    pair_counts = Counter()

    for token_sequence, word_frequency in vocab.items():
        for i in range(len(token_sequence) - 1):
            pair = (token_sequence[i], token_sequence[i + 1])
            pair_counts[pair] += word_frequency

    return pair_counts

pair_counts = get_pair_frequencies(vocab)

for pair, frequency in pair_counts.most_common():
    print(pair, ":", frequency)

The initial counts are:

| Pair | Weighted frequency |
|---|---:|
| `(p, u)` | 17 |
| `(u, n)` | 16 |
| `(u, g)` | 15 |
| `(h, u)` | 10 |
| `(b, u)` | 4 |

Therefore, `(p, u)` is the most frequent pair and is selected for merging.


### 3.3 Merge a selected pair

The next function scans each token sequence from left to right. When it finds the selected adjacent pair, it replaces the pair with their concatenation.



In [ ]:
def merge_pair(pair, vocab):
    new_vocab = {}

    for token_sequence, frequency in vocab.items():
        new_sequence = []
        i = 0

        while i < len(token_sequence):
            if (
                i < len(token_sequence) - 1
                and (token_sequence[i], token_sequence[i + 1]) == pair
            ):
                new_sequence.append(token_sequence[i] + token_sequence[i + 1])
                i += 2
            else:
                new_sequence.append(token_sequence[i])
                i += 1

        new_vocab[tuple(new_sequence)] = frequency

    return new_vocab

best_pair = pair_counts.most_common(1)[0][0]
print("Selected pair:", best_pair)

vocab = merge_pair(best_pair, vocab)

for sequence, frequency in vocab.items():
    print(sequence, ":", frequency)

After the first merge, `p u` becomes `pu`:

- `hug` remains `h u g`
- `pug` becomes `pu g`
- `pun` becomes `pu n`
- `bun` remains `b u n`

We then recalculate pair frequencies on the updated corpus. The next most frequent pair is `(pu, n)` with frequency 12.

### 3.4 Train BPE by repeating the operation


In [ ]:
def train_bpe(corpus, num_merges):
    vocab = {tuple(word): frequency for word, frequency in corpus.items()}
    learned_merges = []

    for step in range(num_merges):
        pair_counts = get_pair_frequencies(vocab)

        if not pair_counts:
            break

        best_pair, best_frequency = pair_counts.most_common(1)[0]
        vocab = merge_pair(best_pair, vocab)
        learned_merges.append(best_pair)

        print(f"Merge {step + 1}: {best_pair} (frequency={best_frequency})")
        print("Updated corpus:", vocab)
        print("-" * 60)

    return vocab, learned_merges

final_vocab, learned_merges = train_bpe(corpus, num_merges=5)

print("Learned merge sequence:")
for merge in learned_merges:
    print(merge)

### What the implementation demonstrates

- `get_pair_frequencies()` counts adjacent pairs and weights them by each word's corpus frequency.
- `merge_pair()` applies one merge to every word.
- `most_common(1)` selects the pair with the largest weighted count.
- `train_bpe()` repeats this process and stores the merge sequence.

**This is a simplified teaching implementation**, not a complete production tokenizer. It uses character-level initialization and omits details such as end-of-word markers, byte encoding, special-token handling, and implementation-specific tie-breaking.

### 3.5 How is BPE used at inference time?

During training, BPE learns an ordered list of merge rules. During tokenization, the learned rules are applied to new text according to the tokenizer's algorithm and preprocessing conventions. The tokenizer does not relearn merges for every sentence.

A key distinction: the simple code above demonstrates the merge-learning idea; production tokenizers may use byte-level representations and additional rules.


## 4. WordPiece and Unigram

These are other subword-tokenization approaches. For this lecture, focus on their central differences rather than implementing each training algorithm.

### 4.1 WordPiece

WordPiece is associated with BERT. It learns a subword vocabulary using a likelihood-based criterion. At tokenization time, it commonly uses a longest-match-first procedure to segment a word into vocabulary pieces.

A continuation piece is often prefixed with `##` in BERT-style token displays. For example, a possible segmentation is:

`playing` → `play` + `##ing`

The prefix `##` indicates that the piece continues a word; it is not necessarily literal text in the original sentence.

### 4.2 Unigram language-model tokenization

Unigram starts from a relatively large candidate vocabulary and iteratively removes pieces that contribute least to the objective. It assigns probabilities to pieces.

For a segmentation \(X=(t_1,\ldots,t_k)\), a simplified independent-piece model has:

\[
P(X)=\prod_{i=1}^{k}P(t_i).
\]

The tokenizer selects a high-probability segmentation under its model. Unigram tokenization is commonly associated with SentencePiece.

### 4.3 Comparison

| Algorithm | Main idea | Common association |
|---|---|---|
| BPE | Iteratively merge frequent adjacent pairs | GPT-2 and many other models |
| WordPiece | Learn a vocabulary with a likelihood-based criterion; longest-match segmentation | BERT |
| Unigram | Start with candidate pieces and prune vocabulary probabilistically | SentencePiece; e.g. T5 tokenizer family |

Model families and implementations can have additional details. Do not assume that every model described as a Transformer uses the same tokenizer.


## 5. Hugging Face: use a pretrained tokenizer

The `transformers` library provides `AutoTokenizer`, which loads the tokenizer configuration and vocabulary associated with a model checkpoint.

We begin with `bert-base-uncased`. The first call may download files from the Hugging Face Hub.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(tokenizer)
print("Tokenizer class:", type(tokenizer))
print("Fast tokenizer:", tokenizer.is_fast)

### 5.1 Tokenize text

`tokenize()` returns token strings. It does not return the integer IDs yet.


In [ ]:
text = "I love Natural Language Processing!"

tokens = tokenizer.tokenize(text)

print("Text:", text)
print("Tokens:", tokens)

Because `bert-base-uncased` is uncased, its tokenizer lowercases text as part of its normalization. Compare it with a cased BERT tokenizer:


In [ ]:
text = "ChatGPT IS AMAZING!"

uncased = AutoTokenizer.from_pretrained("bert-base-uncased")
cased = AutoTokenizer.from_pretrained("bert-base-cased")

print("Uncased:", uncased.tokenize(text))
print("Cased:  ", cased.tokenize(text))

### 5.2 Convert tokens to IDs

A tokenizer has a vocabulary mapping token strings to integer IDs. If the vocabulary has size \(|V|\), token IDs are integers in the range \(0,\ldots,|V|-1\).

\[
f: V \rightarrow \{0,1,\ldots,|V|-1\}.
\]


In [ ]:
text = "I love machine learning"

tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Tokens:", tokens)
print("Token IDs:", token_ids)

vocab = tokenizer.get_vocab()
print("Vocabulary size:", len(vocab))
print("ID for 'love':", vocab.get("love"))

### 5.3 `encode()` and special tokens

`encode()` combines tokenization and conversion to IDs. By default, it adds the special tokens required by the tokenizer's model conventions.

For BERT, a single sequence is commonly formatted as:

`[CLS] sentence tokens [SEP]`


In [ ]:
text = "I love machine learning"

encoded_ids = tokenizer.encode(text)

print("Encoded IDs:", encoded_ids)
print("CLS token:", tokenizer.cls_token, tokenizer.cls_token_id)
print("SEP token:", tokenizer.sep_token, tokenizer.sep_token_id)
print("PAD token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("UNK token:", tokenizer.unk_token, tokenizer.unk_token_id)
print("MASK token:", tokenizer.mask_token, tokenizer.mask_token_id)

Common BERT special tokens:

- `[CLS]`: sequence-level classification representation in traditional BERT use.
- `[SEP]`: separates sequences or marks a sequence boundary.
- `[PAD]`: padding position.
- `[UNK]`: unknown token.
- `[MASK]`: masking symbol used in masked-language-model training.

Other architectures use different special tokens and formatting conventions.


### 5.4 Decode IDs back into text

`decode()` converts a sequence of IDs back into readable text. Decoding may normalize spacing and may optionally remove special tokens.


In [ ]:
text = "I love machine learning"

ids = tokenizer.encode(text)
tokens = tokenizer.convert_ids_to_tokens(ids)

print("IDs:", ids)
print("Tokens:", tokens)
print("Decoded:", tokenizer.decode(ids))
print("Decoded without special tokens:",
      tokenizer.decode(ids, skip_special_tokens=True))

### 5.5 Inspect the complete tokenizer output

Calling the tokenizer directly returns a structured encoding. Depending on the model, it may contain `input_ids`, `token_type_ids`, and `attention_mask`.


In [ ]:
text = "I love machine learning"

encoding = tokenizer(text)

for key, value in encoding.items():
    print(key, ":", value)

#### Interpret the fields

**`input_ids`**

The token IDs. For a single unbatched example, this is a sequence:

\[
\mathbf{x}=(x_1,x_2,\ldots,x_L),\quad x_i\in\{0,\ldots,|V|-1\}.
\]

**`attention_mask`**

A mask that distinguishes real tokens from padding in the usual Hugging Face convention:

\[
m_i =
\begin{cases}
1 & \text{real token}\\
0 & \text{padding}
\end{cases}
\]

**`token_type_ids`**

Some models, including BERT, use segment IDs to distinguish sentence A from sentence B in paired inputs. Many modern architectures do not use this field.


### 5.6 Padding, truncation, and batches

Neural networks commonly process batches represented by rectangular tensors. If examples have different token counts, padding can make their lengths equal.

Let:
- \(B\) be batch size.
- \(L\) be the padded sequence length.

Then the input-ID tensor has shape:

\[
X\in\mathbb{N}^{B\times L}.
\]



In [ ]:
texts = [
    "I love machine learning",
    "Transformers are powerful models"
]

batch = tokenizer(
    texts,
    padding=True,          # pad to longest sequence in this batch
    truncation=True,       # truncate if a maximum length is exceeded
    return_tensors="pt"    # return PyTorch tensors
)

print("input_ids:")
print(batch["input_ids"])

print("\\nattention_mask:")
print(batch["attention_mask"])

print("Shape:", batch["input_ids"].shape)

There are two useful padding modes:

- `padding=True`: pad to the longest sequence in the current batch.
- `padding="max_length"`: pad to the specified `max_length`.

`truncation=True` truncates sequences that exceed the allowed length.


In [ ]:
fixed_batch = tokenizer(
    texts,
    padding="max_length",
    truncation=True,
    max_length=12,
    return_tensors="pt"
)

print(fixed_batch["input_ids"])
print(fixed_batch["attention_mask"])
print("Shape:", fixed_batch["input_ids"].shape)

For the fixed-length example, the shape is:

\[
X\in\mathbb{N}^{2\times 12}.
\]

Padding tokens have attention-mask value 0; actual tokens have value 1. The exact token IDs depend on the tokenizer.


### 5.7 Token offsets

Fast tokenizers can return character offsets linking each token to a span in the original input. An offset is represented as \((s_i,e_i)\), where the end index is exclusive.

For example, if the token `love` has offset `(2, 6)`, then `text[2:6]` returns `love`.

Offsets are useful for mapping token-level predictions back to the original text, such as in named-entity recognition.


In [ ]:
fast_tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased",
    use_fast=True
)

text = "I love machine learning."

encoding = fast_tokenizer(
    text,
    return_offsets_mapping=True
)

tokens = fast_tokenizer.convert_ids_to_tokens(encoding["input_ids"])
offsets = encoding["offset_mapping"]

for token, offset in zip(tokens, offsets):
    start, end = offset
    print(f"{token:>12}  {offset}  {text[start:end]!r}")

Special tokens generally have `(0, 0)` offsets because they do not correspond to a span of the original input. Offset behavior depends on tokenizer normalization and alignment details.


## 6. Train a byte-level BPE tokenizer with Hugging Face `tokenizers`

So far, we have loaded an existing tokenizer. Now we will train a small tokenizer from a text corpus.

**This trains only the tokenizer**, not a language model.

### 6.1 Create a tiny corpus

In a real project, a tokenizer should be trained on a large and representative corpus. The small corpus below is only for demonstration.


In [ ]:
corpus_texts = [
    "I love machine learning.",
    "Machine learning is transforming technology.",
    "Deep learning is a subset of machine learning.",
    "Transformers are powerful neural network architectures.",
    "Natural language processing helps computers understand text.",
    "Language models learn statistical patterns from text.",
    "Tokenization converts text into tokens.",
    "Subword tokenization can represent rare words."
]

corpus_path = "corpus.txt"

with open(corpus_path, "w", encoding="utf-8") as f:
    f.write("\\n".join(corpus_texts))

print(f"Saved {len(corpus_texts)} lines to {corpus_path}")

### 6.2 Configure and train BPE

We use:
- `BPE`: the tokenizer model.
- `NFKC`: a Unicode normalization strategy.
- `ByteLevel`: byte-level pre-tokenization.
- `BpeTrainer`: the component that learns the vocabulary and merge rules.


In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.normalizers import NFKC
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

# 1. Initialize the BPE model.
bpe_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

# 2. Configure Unicode normalization.
bpe_tokenizer.normalizer = NFKC()

# 3. Configure byte-level pre-tokenization.
bpe_tokenizer.pre_tokenizer = ByteLevel(
    add_prefix_space=False
)

# 4. Configure the trainer.
trainer = BpeTrainer(
    vocab_size=500,
    min_frequency=2,
    special_tokens=[
        "[PAD]",
        "[UNK]",
        "[CLS]",
        "[SEP]",
        "[MASK]"
    ]
)

# 5. Train on the text file.
bpe_tokenizer.train(
    files=[corpus_path],
    trainer=trainer
)

# 6. Configure decoding.
bpe_tokenizer.decoder = ByteLevelDecoder()

# 7. Save the complete tokenizer configuration.
bpe_tokenizer.save("my_tokenizer.json")

print("Tokenizer trained and saved.")

### 6.3 Test the trained tokenizer

In [ ]:
test_text = "Transformers are powerful!"

result = bpe_tokenizer.encode(test_text)

print("Input:", test_text)
print("Tokens:", result.tokens)
print("IDs:", result.ids)
print("Decoded:", bpe_tokenizer.decode(result.ids))

### 6.4 Inspect the vocabulary

In [ ]:
learned_vocab = bpe_tokenizer.get_vocab()

print("Learned vocabulary size:", len(learned_vocab))

# Display a sample of token-to-ID entries.
for token, token_id in list(learned_vocab.items())[:30]:
    print(f"{token!r}: {token_id}")

### 6.5 Load the tokenizer through Transformers

A tokenizer saved in the `tokenizers` JSON format can be wrapped in `PreTrainedTokenizerFast` so that it uses the familiar Transformers API.


In [ ]:
from transformers import PreTrainedTokenizerFast

hf_bpe_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="my_tokenizer.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    mask_token="[MASK]"
)

sample = "Transformers are powerful!"

print("Tokens:", hf_bpe_tokenizer.tokenize(sample))
print("IDs:", hf_bpe_tokenizer.encode(sample))
print("Decoded:", hf_bpe_tokenizer.decode(
    hf_bpe_tokenizer.encode(sample)
))

**Caution:** This tiny corpus and vocabulary size are for demonstration only. The resulting tokenizer will have poor coverage and should not be used to train a serious language model. For a real model, the tokenizer's training data, normalization, special tokens, vocabulary size, and pre-tokenization must be designed consistently with the model's training pipeline.


## 7. Compare tokenizers from real model families

Different models can tokenize the same text differently. This affects token count, context-window usage, and the sequence the model receives.

The examples below load three tokenizer families:
- BERT: WordPiece.
- GPT-2: byte-level BPE.
- FLAN-T5: SentencePiece-based Unigram tokenizer.

The first run may download tokenizer files.


In [ ]:
from transformers import AutoTokenizer

model_names = [
    "bert-base-uncased",
    "gpt2",
    "google/flan-t5-small"
]

comparison_text = "Tokenization is fundamental to language models."

for model_name in model_names:
    model_tokenizer = AutoTokenizer.from_pretrained(model_name)

    tokens = model_tokenizer.tokenize(comparison_text)
    ids = model_tokenizer.convert_tokens_to_ids(tokens)

    print("\\nModel:", model_name)
    print("Tokens:", tokens)
    print("IDs:", ids)
    print("Number of tokens:", len(tokens))

The token counts and segmentations may differ. There is no universal tokenization independent of the model: the tokenizer is part of the model's input pipeline.

Also, token count is not the same as word count. A single word can become multiple tokens, and punctuation or whitespace can be represented in tokenizer-specific ways.


## 8. From token IDs to embeddings

A tokenizer returns **discrete integer IDs**. A neural network's embedding layer maps each ID to a learned dense vector.

Let:
- \(V\): vocabulary size.
- \(d\): embedding dimension.
- \(L\): sequence length.
- \(B\): batch size.

The embedding matrix is:

\[
E\in\mathbb{R}^{V\times d}.
\]

A sequence of token IDs is:

\[
\mathbf{x}\in\mathbb{N}^{L}.
\]

Looking up the corresponding rows of \(E\) gives:

\[
H=E[\mathbf{x}]\in\mathbb{R}^{L\times d}.
\]

For a batch:

\[
X\in\mathbb{N}^{B\times L}
\quad\Longrightarrow\quad
H\in\mathbb{R}^{B\times L\times d}.
\]

The vectors are learned during model training. Tokenization itself does not calculate these vectors.


In [ ]:
import torch
import torch.nn as nn

vocab_size = 10_000
embedding_dim = 128

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim
)

# Batch size B=2, sequence length L=4
input_ids = torch.tensor([
    [10, 25, 43, 89],
    [12, 56, 78, 90]
], dtype=torch.long)

output = embedding(input_ids)

print("Input IDs shape:", input_ids.shape)
print("Embedding matrix shape:", embedding.weight.shape)
print("Output embeddings shape:", output.shape)

Expected shapes:

- Input IDs: \((B,L)=(2,4)\).
- Embedding matrix: \((V,d)=(10000,128)\).
- Output: \((B,L,d)=(2,4,128)\).

`nn.Embedding` performs an indexed row lookup. Its weight matrix is a trainable parameter. In a Transformer, positional information is also added or otherwise incorporated before or within the attention layers, depending on the architecture.


## 9. Exercises

Try these exercises before looking up solutions.

### Exercise 1 — Inspect a pretrained tokenizer
1. Load `bert-base-uncased`.
2. Tokenize five sentences of your choice.
3. Display tokens and IDs.
4. Decode the IDs.
5. Compare whitespace-separated word count with token count.

### Exercise 2 — Compare tokenizer families
1. Compare BERT, GPT-2, and FLAN-T5 on the same five sentences.
2. Which tokenizer produces the most tokens for each sentence?
3. How do punctuation, capitalization, and unfamiliar words affect the output?
4. Explain why token count matters for a model's context window.

### Exercise 3 — Train a tokenizer
1. Prepare a text corpus in a domain of your choice.
2. Train a BPE tokenizer.
3. Inspect its vocabulary.
4. Test it on sentences that were not in the training corpus.
5. Explain why a tiny training corpus is not sufficient for a production tokenizer.

### Exercise 4 — Understand padding
Create a batch containing three sentences with different lengths. Print `input_ids` and `attention_mask`. Explain which positions are padding.

### Exercise 5 — Connect IDs to embeddings
Create an embedding layer with vocabulary size 20,000 and dimension 256. For a batch with shape \((B,L)=(3,10)\), state and verify the output shape.


## 10. Key takeaways

1. **Preprocessing** transforms raw text, but must be chosen carefully to avoid removing useful information.
2. **Tokenization** converts text into discrete units.
3. **Subword tokenization** balances vocabulary size and sequence length.
4. **BPE** learns merges by repeatedly combining frequent adjacent pairs.
5. **WordPiece** and **Unigram** use different vocabulary-learning and segmentation principles.
6. Hugging Face tokenizers expose useful operations such as `tokenize`, `encode`, `decode`, padding, truncation, and offset mapping.
7. A tokenizer maps text to IDs; an embedding layer maps IDs to vectors.

### Transition to Lecture 2

The next question is:

> Once each token has an integer ID, how can a neural network represent it as a useful vector?

That leads to **embeddings and word representations**.
